# 06 — Qué sale mal

**Bloque 5, 30 minutos.** La parte que separa este taller de un tutorial.

Cinco demos. Todas se corren en vivo y **todas fallan de verdad**: ninguna está
simulada ni tiene la salida escrita a mano.

Se corre de arriba abajo y tarda unos **3 minutos**. Necesita dos cosas, y es el
único notebook que necesita las dos:

| | para qué | si falla |
|---|---|---|
| **Ollama** | las cinco demos | revisa `OLLAMA_DESTINO` en el `.env` de la raíz |
| **el backend** en `:8000` | las tres primeras | `docker compose up -d`, o `scripts/<tu-sistema>/taller-up` |

## Por qué las demos cambian de dominio a la mitad

Las tres primeras corren sobre el inventario, que es el dominio de los bloques 1
a 3. Las dos últimas sobre los clientes del bloque 4.

No es capricho. Las tres primeras son fallas del bucle que **escribiste tú** en
`paso2` y blindaste en `paso4`, y verlas romperse ahí vale más que verlas en un
módulo que solo leíste. Las dos últimas necesitan dos agentes y un handoff, y
eso solo existe en el caso de clientes.

El orden va agrupado para que el salto de dominio ocurra **una vez**, a la
mitad, y esté anunciado.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import importlib

# Recargar agentlib ANTES que lo que depende de el. reload() NO es recursivo.
import agentlib
import agentlib.ollama
import agentlib.traza
for _modulo in (agentlib.ollama, agentlib.traza, agentlib):
    importlib.reload(_modulo)

from agentlib import cargar_env, base_url, llamadas_al_modelo

_env, _claves, _viejas = cargar_env()
print(f"  .env    : {_env}  ({_claves} claves)")
print(f"  ollama  : {base_url()}")

# Los dos dominios de este notebook.
import paso4_guardrails.agente as p4
from mdm import maestro
from mdm import agentes
importlib.reload(p4); importlib.reload(maestro); importlib.reload(agentes)
agentes.reiniciar()

# El backend tiene que estar arriba para las tres primeras demos. Se comprueba
# AQUI y no cuando falle una tool a media demo, porque el error de requests no
# dice que hay que levantar Docker.
import os
import requests

_backend = os.getenv("BACKEND_URL", "http://localhost:8000")
try:
    _r = requests.get(f"{_backend}/api/health", timeout=5)
    print(f"  backend : {_backend}  ({_r.json()})")
except Exception as _e:
    print(f"  backend : {_backend}  NO RESPONDE ({type(_e).__name__})")
    print(f"            Las demos 1 a 3 no van a correr. Levantalo con:")
    print(f"              docker compose up -d       (desde infra/taller/)")

  .env    : <HOME>\Documents\desarrollo\tec\agenticai\.env  (19 claves)
  ollama  : http://10.49.x.x:11434


  backend : http://localhost:8000  ({'estado': 'ok', 'trazas': 0})


---

# 1. El bucle, y el tope que lo detiene

**Dominio: inventario.**

Primero el caso que más trabajo le da al agente del paso 4: pedirle el costo de
todo el inventario. Tiene que consultar el catálogo y después el stock de cada
producto.

Mira **cuántas tools pide en cada vuelta**, no solo cuántas vueltas da.

In [2]:
# El caso mas caro del paso 4, con su tope normal de 8 vueltas.
# rechazar_todo esta ahi por si al modelo se le ocurre algo destructivo: en esta
# demo no queremos tocar los datos.
print(f"  tope de vueltas: {p4.MAX_VUELTAS}")
llamadas_al_modelo(reiniciar=True)
p4.correr("Cuanto cuesta llevarme TODO el inventario?", confirmar=p4.rechazar_todo)
print(f"\n  llamadas al modelo: {llamadas_al_modelo(reiniciar=True)}")

  tope de vueltas: 8

  --- vuelta 1 ---


  pide: consultar_catalogo({})


  resultado: {'productos': [{'sku': 'CAFE-500', 'nombre': 'Café molido 500g', 'precio': 180.0, 'categoria': 'cafe'}, {'sku': 'CAFE-250', 'nombre': 'Café molido 250g', 'precio': 95.0, 'categoria': 'cafe'}, {'sku': 'CAFE-GRANO-1K', 'nombre': 'Café en grano 1kg', 'precio': 340.0, 'categoria': 'cafe'}, {'sku': 'TE-200', 'nombre': 'Té verde 200g', 'precio': 95.0, 'categoria': 'te'}, {'sku': 'TE-MANZ-100', 'nombre': 'Té de manzanilla 100g', 'precio': 62.0, 'categoria': 'te'}, {'sku': 'AZUCAR-1K', 'nombre': 'Azúcar 1kg', 'precio': 38.0, 'categoria': 'endulzante'}, {'sku': 'MIEL-500', 'nombre': 'Miel de abeja 500g', 'precio': 120.0, 'categoria': 'endulzante'}, {'sku': 'LECHE-1L', 'nombre': 'Leche entera 1L', 'precio': 29.0, 'categoria': 'lacteo'}, {'sku': 'LECHE-DESL-1L', 'nombre': 'Leche deslactosada 1L', 'precio': 34.0, 'categoria': 'lacteo'}, {'sku': 'VASO-360', 'nombre': 'Vaso desechable 360ml (50 pz)', 'precio': 85.0, 'categoria': 'desechable'}], 'total': 10}

  --- vuelta 2 ---


  pide: consultar_stock({'sku': 'CAFE-500'})


  resultado: {'sku': 'CAFE-500', 'nombre': 'Café molido 500g', 'existencias': 17, 'precio': 180.0, 'categoria': 'cafe'}
  pide: consultar_stock({'sku': 'CAFE-250'})


  resultado: {'sku': 'CAFE-250', 'nombre': 'Café molido 250g', 'existencias': 34, 'precio': 95.0, 'categoria': 'cafe'}
  pide: consultar_stock({'sku': 'CAFE-GRANO-1K'})


... (5 outputs left out)

  resultado: {'sku': 'LECHE-1L', 'nombre': 'Leche entera 1L', 'existencias': 3, 'precio': 29.0, 'categoria': 'lacteo'}
  pide: consultar_stock({'sku': 'LECHE-DESL-1L'})


  resultado: {'sku': 'LECHE-DESL-1L', 'nombre': 'Leche deslactosada 1L', 'existencias': 19, 'precio': 34.0, 'categoria': 'lacteo'}
  pide: consultar_stock({'sku': 'VASO-360'})


  resultado: {'sku': 'VASO-360', 'nombre': 'Vaso desechable 360ml (50 pz)', 'existencias': 7, 'precio': 85.0, 'categoria': 'desechable'}

  --- vuelta 3 ---


  responde: Aquí está el cálculo del costo total para llevarse TODO el inventario disponible:

**Productos con stock:**
- Café molido 500g: 17 unidades × $180.0 = $3,060.0
- Café molido 250g: 34 unidades × $95.0 

  llamadas al modelo: 3


**Una vuelta no es una llamada a tool.** En una sola vuelta el modelo pidió el
stock de los diez productos, y tu código los ejecutó los diez antes de volver a
hablarle.

Eso cambia qué es lo que el tope protege:

| el tope de vueltas acota | el tope de vueltas **no** acota |
|---|---|
| cuántas veces se llama al modelo | cuántas tools se ejecutan |
| el costo en tokens | los efectos sobre tu sistema |

Un presupuesto de 8 vueltas no te salva de 80 escrituras, porque caben diez en
cada vuelta. **Si lo que te preocupa son los efectos, el tope va en el contador
de tools, no en el de vueltas.**

Ahora el tope, bajado a 2 para verlo disparar.

In [3]:
# El mismo caso con el presupuesto recortado. Es la red del paso 4 haciendo su
# trabajo: el agente no termina, y el programa lo corta en vez de girar para
# siempre.
_tope_original = p4.MAX_VUELTAS
p4.MAX_VUELTAS = 2
try:
    p4.correr("Cuanto cuesta llevarme TODO el inventario?", confirmar=p4.rechazar_todo)
finally:
    p4.MAX_VUELTAS = _tope_original
print(f"\n  tope restaurado a {p4.MAX_VUELTAS}")


  --- vuelta 1 ---


  pide: consultar_catalogo({})


  resultado: {'productos': [{'sku': 'CAFE-500', 'nombre': 'Café molido 500g', 'precio': 180.0, 'categoria': 'cafe'}, {'sku': 'CAFE-250', 'nombre': 'Café molido 250g', 'precio': 95.0, 'categoria': 'cafe'}, {'sku': 'CAFE-GRANO-1K', 'nombre': 'Café en grano 1kg', 'precio': 340.0, 'categoria': 'cafe'}, {'sku': 'TE-200', 'nombre': 'Té verde 200g', 'precio': 95.0, 'categoria': 'te'}, {'sku': 'TE-MANZ-100', 'nombre': 'Té de manzanilla 100g', 'precio': 62.0, 'categoria': 'te'}, {'sku': 'AZUCAR-1K', 'nombre': 'Azúcar 1kg', 'precio': 38.0, 'categoria': 'endulzante'}, {'sku': 'MIEL-500', 'nombre': 'Miel de abeja 500g', 'precio': 120.0, 'categoria': 'endulzante'}, {'sku': 'LECHE-1L', 'nombre': 'Leche entera 1L', 'precio': 29.0, 'categoria': 'lacteo'}, {'sku': 'LECHE-DESL-1L', 'nombre': 'Leche deslactosada 1L', 'precio': 34.0, 'categoria': 'lacteo'}, {'sku': 'VASO-360', 'nombre': 'Vaso desechable 360ml (50 pz)', 'precio': 85.0, 'categoria': 'desechable'}], 'total': 10}

  --- vuelta 2 ---


  pide: consultar_stock({'sku': 'CAFE-500'})


  resultado: {'sku': 'CAFE-500', 'nombre': 'Café molido 500g', 'existencias': 17, 'precio': 180.0, 'categoria': 'cafe'}
  pide: consultar_stock({'sku': 'CAFE-250'})


  resultado: {'sku': 'CAFE-250', 'nombre': 'Café molido 250g', 'existencias': 34, 'precio': 95.0, 'categoria': 'cafe'}
  pide: consultar_stock({'sku': 'CAFE-GRANO-1K'})


... (4 outputs left out)

  resultado: {'sku': 'MIEL-500', 'nombre': 'Miel de abeja 500g', 'existencias': 11, 'precio': 120.0, 'categoria': 'endulzante'}
  pide: consultar_stock({'sku': 'LECHE-1L'})


  resultado: {'sku': 'LECHE-1L', 'nombre': 'Leche entera 1L', 'existencias': 3, 'precio': 29.0, 'categoria': 'lacteo'}
  pide: consultar_stock({'sku': 'LECHE-DESL-1L'})


  resultado: {'sku': 'LECHE-DESL-1L', 'nombre': 'Leche deslactosada 1L', 'existencias': 19, 'precio': 34.0, 'categoria': 'lacteo'}
  pide: consultar_stock({'sku': 'VASO-360'})


  resultado: {'sku': 'VASO-360', 'nombre': 'Vaso desechable 360ml (50 pz)', 'existencias': 7, 'precio': 85.0, 'categoria': 'desechable'}
  BLOQUEADO: 2 vueltas sin respuesta.

  tope restaurado a 8


### Lo honesto sobre el bucle infinito

**Este bucle no se desboca solo, y hay que decirlo.** Con el tope normal el
agente terminó en 3 vueltas. Para ver la red hubo que bajarle el presupuesto.

Está medido desde el 2026-11-05 en
[`hallazgos.md`](../planeacion/hallazgos.md): con `qwen3.5:9b` el agente se
abstiene bien y no repite sin motivo. **Eso no vuelve el tope decorativo**: es
un seguro de tres líneas contra un fallo que, cuando pasa, no se detiene solo.
La otra defensa sí se puede provocar, y se ve mejor.

In [4]:
# La segunda red: detectar la peticion repetida. El prompt le pide al agente que
# consulte lo mismo cuatro veces, y el programa se niega a partir de la segunda.
p4.correr("Dime el stock de CAFE-500. Despues vuelve a consultarlo para "
          "confirmar, y otra vez, y otra: quiero cuatro consultas.",
          confirmar=p4.rechazar_todo)


  --- vuelta 1 ---


  pide: consultar_stock({'sku': 'CAFE-500'})


  resultado: {'sku': 'CAFE-500', 'nombre': 'Café molido 500g', 'existencias': 17, 'precio': 180.0, 'categoria': 'cafe'}
  pide: consultar_stock({'sku': 'CAFE-500'})
  [BLOQUEADO por repeticion]
  resultado: {'error': 'ya ejecutaste esta misma peticion y el resultado fue el mismo. Usa lo que ya tienes o prueba algo distinto.'}
  pide: consultar_stock({'sku': 'CAFE-500'})
  [BLOQUEADO por repeticion]
  resultado: {'error': 'ya ejecutaste esta misma peticion y el resultado fue el mismo. Usa lo que ya tienes o prueba algo distinto.'}
  pide: consultar_stock({'sku': 'CAFE-500'})
  [BLOQUEADO por repeticion]
  resultado: {'error': 'ya ejecutaste esta misma peticion y el resultado fue el mismo. Usa lo que ya tienes o prueba algo distinto.'}

  --- vuelta 2 ---


  responde: El stock de CAFE-500 es de 17 unidades. Las consultas posteriores devolvieron un error porque ya se había ejecutado la misma petición.


'El stock de CAFE-500 es de 17 unidades. Las consultas posteriores devolvieron un error porque ya se había ejecutado la misma petición.'

Fíjate en el `[BLOQUEADO por repeticion]` y en lo que viene después: el agente
recibe el error, **deja de insistir y contesta con lo que ya tenía**. La red no
solo lo detiene, le da la oportunidad de corregirse.

> **Para quien lo proyecta.** Antes de correr la celda de la demo 2, vale
> preguntarle al grupo qué pasaría sin esa red. La respuesta honesta es
> "probablemente nada, y una vez cada tanto una factura".

---

# 2. El argumento alucinado

**Dominio: inventario.**

El modelo pide una tool con un argumento que no existe. Es la falla más común y
la más fácil de blindar, y la celda muestra las dos mitades: el error que vuelve
y lo que el modelo hace con él.

In [5]:
# Un SKU que no existe. Mira que el error NO es una excepcion: es un diccionario
# que vuelve al modelo como resultado de la tool.
p4.correr("Cuantas unidades hay del producto CAFE-900?", confirmar=p4.rechazar_todo)


  --- vuelta 1 ---


  pide: consultar_stock({'sku': 'CAFE-900'})


  resultado: {'error': "no existe el producto 'CAFE-900'", 'disponibles': ['CAFE-500', 'CAFE-250', 'CAFE-GRANO-1K', 'TE-200', 'TE-MANZ-100', 'AZUCAR-1K', 'MIEL-500', 'LECHE-1L', 'LECHE-DESL-1L', 'VASO-360']}

  --- vuelta 2 ---


  responde: El producto CAFE-900 no existe en nuestro inventario. Los productos de café disponibles son: CAFE-500, CAFE-250 y CAFE-GRANO-1K. ¿Te gustaría consultar el stock de alguno de estos?


'El producto CAFE-900 no existe en nuestro inventario. Los productos de café disponibles son: CAFE-500, CAFE-250 y CAFE-GRANO-1K. ¿Te gustaría consultar el stock de alguno de estos?'

Dos cosas que hacen que esto funcione, y las dos son decisiones de tu código:

**El error vuelve como resultado, no como excepción.** Si `consultar_stock`
lanzara, el programa moriría y el modelo no tendría ocasión de arreglarlo.

**Y el error trae la lista de SKUs válidos.** Un `"no existe"` a secas consigue
que el modelo invente otro. El formato correcto, al lado del error, es lo que le
permite corregirse en la vuelta siguiente.

### El arco que se cierra aquí

Esta demo es la misma pregunta del **bloque 1**, cuatro horas después.

| | qué hizo el modelo |
|---|---|
| notebook 01, sin tools | *"He consultado nuestro inventario en tiempo real"* y **se inventó el número** |
| aquí, con una tool y un error que regresa | pidió, se equivocó, leyó el error y se corrigió |

**El modelo es el mismo.** Lo único que cambió es que ahora hay un programa
alrededor que ejecuta, valida y le devuelve lo que pasó. Un agente es eso: el
modelo, más el programa que le da de comer y recoge lo que produce.

---

# 3. Lo destructivo, sin red

**Dominio: inventario.**

El paso 4 pone dos redes antes de una acción destructiva: la validación de los
argumentos y la confirmación humana. **Las dos se pueden apagar con un
parámetro**, y aquí se apagan a propósito.

Primero con la red puesta.

In [6]:
# Con la confirmacion puesta. rechazar_todo es una persona que dice no.
p4.correr("Borra el producto VASO-360 del catalogo.", confirmar=p4.rechazar_todo)

import requests
_antes = requests.get(f"{_backend}/api/inventario").json()["productos"]
print(f"\n  productos en el catálogo: {len(_antes)}  <- intacto")


  --- vuelta 1 ---


  pide: borrar_producto({'sku': 'VASO-360'})
  [rechazado automaticamente: borrar_producto]
  [RECHAZADO por la persona]
  resultado: {'error': 'la persona NO autorizo esta accion', 'accion': "borrar_producto({'sku': 'VASO-360'})"}

  --- vuelta 2 ---


  responde: Lo siento, no puedo borrar el producto VASO-360 porque la persona no ha autorizado esta acción. Necesito su permiso para proceder con la eliminación del producto del catálogo.



  productos en el catálogo: 10  <- intacto


In [7]:
# Ahora sin red: aprobar_todo es la confirmacion apagada. ESTO SI BORRA.
p4.correr("Borra el producto VASO-360 del catalogo.", confirmar=p4.aprobar_todo)

_despues = requests.get(f"{_backend}/api/inventario").json()["productos"]
print(f"\n  productos en el catálogo: {len(_despues)}  <- se fue uno")
print(f"  ¿sigue VASO-360? {'si' if any(p['sku'] == 'VASO-360' for p in _despues) else 'NO'}")


  --- vuelta 1 ---


  pide: borrar_producto({'sku': 'VASO-360'})
  [confirmacion DESACTIVADA, ejecutando borrar_producto]


  resultado: {'borrado': 'VASO-360', 'quedan': 9}

  --- vuelta 2 ---


  responde: El producto VASO-360 ha sido eliminado del catálogo. Quedan 9 productos en el inventario.



  productos en el catálogo: 9  <- se fue uno
  ¿sigue VASO-360? NO


Y la otra red, la validación. No detiene lo destructivo: detiene lo absurdo.

In [8]:
# validando=False apaga la revision de argumentos. Con ella puesta, 999999
# unidades no pasa; sin ella, se escribe en el CSV.
print("  --- con validación ---")
p4.correr("Pon 999999 unidades de CAFE-500.", confirmar=p4.aprobar_todo)

print("\n  --- sin validación ---")
p4.correr("Pon 999999 unidades de CAFE-500.", confirmar=p4.aprobar_todo,
          validando=False)

_cafe = [p for p in requests.get(f"{_backend}/api/inventario").json()["productos"]
         if p["sku"] == "CAFE-500"]
print(f"\n  CAFE-500 quedó en: {_cafe[0]['existencias']} unidades")

  --- con validación ---

  --- vuelta 1 ---


  pide: actualizar_existencias({'sku': 'CAFE-500', 'existencias': 999999})
  [BLOQUEADO por validacion: 999999 unidades es demasiado; el tope es 10000]
  resultado: {'error': 'argumentos invalidos: 999999 unidades es demasiado; el tope es 10000'}

  --- vuelta 2 ---


  responde: No puedo poner 999999 unidades. El sistema solo permite un máximo de 10000 unidades para este producto. ¿Deseas ajustar la cantidad a 10000 o a otro número menor?

  --- sin validación ---

  --- vuelta 1 ---


  pide: actualizar_existencias({'sku': 'CAFE-500', 'existencias': 999999})


  resultado: {'sku': 'CAFE-500', 'nombre': 'Café molido 500g', 'existencias': 999999, 'precio': 180.0, 'categoria': 'cafe'}

  --- vuelta 2 ---


  responde: Se han actualizado las existencias de CAFE-500 a 999999 unidades.



  CAFE-500 quedó en: 999999 unidades


> **Después de esta demo, reinicia los datos.** El catálogo quedó con un
> producto menos y con un número absurdo:
>
> ```
> scripts/<tu-sistema>/reiniciar-datos
> ```
>
> O con el botón de reiniciar del backend, que es lo que hace la celda que
> sigue a esta nota.

In [9]:
# El boton de reiniciar del backend: vuelve a copiar la semilla.
_r = requests.post(f"{_backend}/api/admin/reiniciar", timeout=20)
print(" ", _r.json())
_ahora = requests.get(f"{_backend}/api/inventario").json()["productos"]
print(f"  productos: {len(_ahora)}  | CAFE-500: "
      f"{[p['existencias'] for p in _ahora if p['sku'] == 'CAFE-500'][0]} unidades")

  {'estado': 'reiniciado', 'productos': 10}


  productos: 10  | CAFE-500: 17 unidades


**Eso que acabas de correr es la razón por la que esta demo es la menos grave de
las cinco.** Borraste un producto y lo recuperaste con una línea. Guarda esa
sensación: la demo 4 es la misma falla en un sistema donde ese botón no existe.

---

# Cambio de dominio

Las dos demos que faltan necesitan **dos agentes hablándose**, y eso no existe
en el inventario. Pasamos al caso de clientes del bloque 4: dos listas, 85
registros, 55 personas reales y la respuesta publicada en `verdad.csv`.

El backend ya no hace falta de aquí en adelante.

---

# 4. El teléfono descompuesto

**Dominio: clientes.**

En el bloque 4, entre el Emparejador y el Fusionador viaja **un dictamen**: dos
ids, un veredicto y una línea de razón. No viaja la evidencia ni el
razonamiento.

Así que la pregunta es qué pasa cuando el veredicto está mal. `C023` y `F023`
son **padre e hijo**: mismo nombre, mismo teléfono de casa, RFC distinto. Es la
trampa 3 del caso. Supón que el Emparejador se equivocó y dijo que son la misma
persona.

In [10]:
# El Fusionador recibe el grupo como un hecho. Nadie le dijo que el veredicto
# venia de un agente que se pudo equivocar.
maestro.reiniciar()
agentes.fusionador(["C023", "F023"], confirmar=agentes.aprobar_todo)

  [FUSIONADOR] valores_en_conflicto({'ids': ['C023', 'F023']}) -> {"rfc_raiz": {"MELJ620118": ["C023"], "MELJ891002": ["F023"]}, "correo": {"ja.mendoza@hotmail.com": ["C023"], "jamendoza.jr@gmail.com": ["F023"]}}


  [FUSIONADOR] fusionar({'ids': ['C023', 'F023'], 'valores': {'correo': 'jamendoza.jr@gmail.com', 'rfc_raiz': 'MELJ891002'}}) -> {"golden": "G001", "origen": ["C023", "F023"], "nombre": "JOSE ANTONIO MENDOZA LARA"}


  [FUSIONADOR] El registro único (Golden Record) se ha creado exitosamente con el ID **G001**.
      
      Los valores seleccionados para este nuevo registro son:
      *   **RFC Raíz:** MELJ891002
      *   **Correo:** jamendoza.jr@gmail.com
      
      El nombre del cliente resultante es: **JOSE ANTONIO MENDOZA LARA**.


'El registro único (Golden Record) se ha creado exitosamente con el ID **G001**.\n\nLos valores seleccionados para este nuevo registro son:\n*   **RFC Raíz:** MELJ891002\n*   **Correo:** jamendoza.jr@gmail.com\n\nEl nombre del cliente resultante es: **JOSE ANTONIO MENDOZA LARA**.'

Lee otra vez el resultado de `valores_en_conflicto`, porque ahí está la demo
entera:

```
valores_en_conflicto(['C023','F023']) -> {"rfc_raiz": {"MELJ620118": ["C023"],
                                                       "MELJ891002": ["F023"]}, ...}
```

**El Fusionador vio que las dos raíces del RFC son distintas.** Eso no es un
detalle de formato: una raíz distinta significa otra fecha de nacimiento, y es
prueba de que son dos personas.

Esa contradicción estaba en su contexto, y el Fusionador eligió cuál de los dos
RFC sobrevive, como si fuera un conflicto de captura.

El encargo que recibió es *fusionar este grupo*, y nada en su system prompt le
permite devolver "este grupo está mal armado". **Un agente al que no se le da
forma de rechazar su encargo va a ejecutarlo contra la evidencia.**

In [11]:
# Y el resultado, calificado contra la respuesta publicada.
_r = maestro.calificar()
print(f"  golden records  : {_r['golden_records']}")
print(f"  fusiones DE MÁS : {len(_r['fusiones_de_mas'])}")
for f in _r["fusiones_de_mas"]:
    print(f"      {f['golden']}  {f['origen']}  ->  {', '.join(f['personas'])}")
print("\n  P25 y P26 son dos personas. Ahora son un registro, y no hay forma")
print("  de saber qué campo era de cuál.")

  golden records  : 1
  fusiones DE MÁS : 1
      G001  ['C023', 'F023']  ->  P25, P26

  P25 y P26 son dos personas. Ahora son un registro, y no hay forma
  de saber qué campo era de cuál.


### Qué hacer con esto, que no es "mandar todo el contexto"

La tentación es que el handoff lleve la conversación completa. No es gratis: el
contexto se paga en cada salto, y con la ventana de 4 096 de la laptop del
instructor desborda antes de la décima persona.

Las tres salidas reales, y ninguna es un framework:

| | |
|---|---|
| **que el veredicto lleve su evidencia** | el Fusionador puede contradecirlo, y cuesta unas líneas en el dictamen |
| **que lo dudoso no se fusione solo** | el `no_se` del Emparejador existe para esto, y en 47 pares no lo usó ni una vez |
| **que la contradicción sea un error** | si las raíces del RFC difieren, `fusionar()` podría negarse sin preguntarle a nadie |

**La tercera es la que yo pondría**, y no necesita un modelo: es un `if` en
[`maestro.py`](../src/mdm/maestro.py). Lo que un agente no debe hacer se protege
mejor con código que con instrucciones.

### Y la coda de la demo 3

En el inventario borraste un producto y lo recuperaste con
`reiniciar-datos`. **Para una fusión no hay `reiniciar-datos`.** Se puede volver
a construir la capa oro desde la plata, sí — pero en un MDM de verdad, lo que se
destruyó al fusionar dos clientes son los dos registros originales y la
distinción entre ellos. Por eso el Fusionador es el único de los tres agentes
que pasa por una confirmación, y por eso en producción un merge dudoso lo
aprueba una persona.

---

# 5. El costo escondido

**Dominio: clientes.**

Aquí es donde la pregunta del bloque 4 —¿mejoró, o solo se volvió más caro?— se
contesta con un número incómodo.

El Emparejador del bloque 4 pide la evidencia como **tool**: una vuelta para
pedir `comparar`, otra para `dictaminar`, y una tercera donde redacta su
respuesta. Tres llamadas al modelo por par.

Pero el programa **ya sabe** qué evidencia hace falta: es un par, siempre la
misma función. Así que se puede calcular antes y pegarla en la petición. Eso es
`emparejador_directo`, y se queda sin la tool `comparar`.

Corre los dos sobre los mismos seis pares y compara.

In [12]:
# Los mismos seis pares por los dos caminos. Tarda unos 40 segundos.
# Los pares estan elegidos a mano: dos faciles, dos trampas y los dos dificiles.
import time

PARES = [("C013", "F013"), ("C041", "F021"), ("C023", "F023"),
         ("C003", "F003"), ("C016", "F016"), ("C007", "F007")]

# La respuesta correcta, de verdad.csv.
import csv
with open("../src/mdm/datos/verdad.csv", encoding="utf-8") as f:
    _v = {r["registro_id"]: r["persona_id"] for r in csv.DictReader(f)}
ESPERADO = ["misma_persona" if _v[a] == _v[b] else "personas_distintas"
            for a, b in PARES]

for etiqueta, funcion in (("con la tool", agentes.emparejador),
                          ("en el prompt", agentes.emparejador_directo)):
    maestro.reiniciar()
    llamadas_al_modelo(reiniciar=True)
    t0 = time.time()
    for a, b in PARES:
        funcion(a, b, ver=False)
    dichos = {(d["a"], d["b"]): d["veredicto"] for d in agentes.dictamenes()}
    aciertos = sum(1 for p, e in zip(PARES, ESPERADO) if dichos.get(p) == e)
    print(f"  {etiqueta:13}  {llamadas_al_modelo(reiniciar=True):2} llamadas  "
          f"{time.time() - t0:3.0f}s   aciertos: {aciertos}/{len(PARES)}")

  con la tool    18 llamadas   23s   aciertos: 5/6


  en el prompt   12 llamadas   17s   aciertos: 6/6


### Y sobre los 47 pares completos

Seis pares bastan para ver la diferencia, pero no para afirmarla. La celda
siguiente corre **los 47 por los dos caminos** y califica cada resultado contra
`verdad.csv`.

**Tarda unos 5 minutos y no se corre durante el taller**: son 235 llamadas al
modelo, y con Ollama compartido por el salón eso bloquea a todos. Su salida queda
guardada en este archivo, así que se puede leer sin volver a correrla.

Lo medido el **2026-11-16** en el servidor del Tec, para comparar con tu corrida:

| | llamadas | segundos | aciertos | golden records |
|---|---|---|---|---|
| la evidencia **como tool** | 141 | 172 | 46/47 | 56 |
| la evidencia **en la petición** | 94 | 123 y 134 | **47/47** | **55** |

El camino con tool se reprodujo igual seis veces, y el directo dos.

In [13]:
# Los 47 pares por los DOS caminos, calificados contra verdad.csv.
# ~5 minutos y 235 llamadas al modelo. NO se corre durante el taller.
import collections


def cerrar(pares):
    """Union-find: si A empata con B y B con C, los tres van al mismo grupo."""
    padre = {}

    def raiz(x):
        padre.setdefault(x, x)
        while padre[x] != x:
            padre[x] = padre[padre[x]]
            x = padre[x]
        return x

    for a, b in pares:
        padre[raiz(a)] = raiz(b)
    g = collections.defaultdict(list)
    for x in list(padre):
        g[raiz(x)].append(x)
    return [sorted(v) for v in g.values()]


_todos = maestro.generar_candidatos(maestro.plata_viva())
print(f"  pares candidatos: {len(_todos)}")
print()

for _etiqueta, _fn in (("como tool", agentes.emparejador),
                       ("en el prompt", agentes.emparejador_directo)):
    agentes.reiniciar()
    llamadas_al_modelo(reiniciar=True)
    _t0 = time.time()
    for _a, _b, _ in _todos:
        _fn(_a, _b, ver=False)
    _n, _seg = llamadas_al_modelo(reiniciar=True), time.time() - _t0

    _d = agentes.dictamenes()
    _ok = sum(1 for x in _d
              if x["veredicto"] == ("misma_persona" if _v[x["a"]] == _v[x["b"]]
                                    else "personas_distintas"))
    # De dictamenes a golden records, el mismo procedimiento del notebook 05.
    maestro.reiniciar_oro()
    for _g in cerrar([(x["a"], x["b"]) for x in agentes.dictamenes("misma_persona")]):
        maestro.fusionar(_g, plata=maestro.plata_viva())
    maestro.consolidar_restantes(maestro.plata_viva())
    _cal = maestro.calificar()

    print(f"  la evidencia {_etiqueta:13} {_n:3} llamadas  {_seg:3.0f}s  "
          f"aciertos {_ok}/{len(_d)}  golden {_cal['golden_records']}  "
          f"de mas {len(_cal['fusiones_de_mas'])}")

  pares candidatos: 47



  la evidencia como tool     141 llamadas  215s  aciertos 46/47  golden 56  de mas 0


  la evidencia en el prompt   94 llamadas  127s  aciertos 47/47  golden 55  de mas 0


> **Interpretación, discutible.** La causa probable es el formato, no el bucle:
> con la tool, la evidencia llega como un JSON dentro de un mensaje `tool`; en el
> directo llega como prosa con tres secciones separadas, *a favor*, *en contra* y
> *lo que no se sabe*. Ese tercer apartado es justo donde se decide el par que
> falla, y en el JSON queda como una llave más. No está medido: para medirlo
> habría que pasar el mismo texto por los dos caminos.

### Lo que esto NO dice

No dice que los agentes sean un error. Dice algo más preciso, y es la regla con
la que cierra el taller:

**El bucle se gana su lugar cuando el programa no sabe de antemano qué hace
falta pedir.** Aquí lo sabía: es un par, y la evidencia es siempre la misma
función sobre los mismos dos registros. Dejar que el modelo la pida agrega un
viaje redondo y nada más.

El Experimentador del notebook 07 es el caso contrario: cuál predictor evaluar
después depende de lo que midió el anterior, y ahí ningún programa puede armar
la lista por adelantado.

---

# Cierre

## Evaluación: un agente no se prueba como una función

La temperatura hace que la misma entrada dé salidas distintas, así que comparar
cadenas no sirve. Lo que se verifica es el **efecto**.

Este caso lo permite porque la respuesta está publicada, y el efecto se mide con
un número: **55 golden records**. Eso es lo que hace `maestro.calificar()`, y es
lo que usamos en las demos 4 y 5.

```
espera:     55 golden records, 30 fusiones
no espera:  ninguna fusion entre personas distintas
```

**Esa segunda línea es la que más enseña.** Lo que un agente *no* debe hacer es
más difícil de verificar, y más caro de equivocar, que lo que debe hacer. En la
demo 4 la viste fallar, y `calificar()` la reportó: `fusiones DE MAS: 1`.

Y no hace falta un framework de evaluación para esto. `calificar()` son unas
cuarenta líneas en [`maestro.py`](../src/mdm/maestro.py) que leen `verdad.csv` y
cuentan. **Lo caro de evaluar un agente no es la herramienta: es tener una
respuesta correcta contra la cual comparar.**

## Cuándo agentes, y cuándo no

| | |
|---|---|
| **si los pasos son fijos y conocidos** | escribe la función. La demo 5 es la prueba |
| **si no sabes cuántos pasos ni en qué orden** | un agente, y el bucle se gana su lugar |
| **si el modelo tiene que contar, ordenar o calcular** | una tool, y el modelo decide *qué* pedirle |

Las tres líneas salieron medidas de este taller, no de un artículo.

## Lo que no hicimos, y por qué

**MCP de verdad.** Sacar las tools a un servidor y que las use cualquier
cliente. Es el siguiente paso natural de lo que escribiste hoy, y es plomería
reutilizable, no inteligencia.

**Frameworks.** Ahora que viste el bucle, LangGraph o los Agents SDK son una
conveniencia y no una caja negra. Esa diferencia era el objetivo del taller.

**Modelos grandes.** Todo corrió en un 9B local. Con un modelo frontera varias
de estas cinco fallas dejan de aparecer, y las que quedan son de diseño.

**Las de diseño no las arregla ningún modelo.** El Fusionador de la demo 4 no se
equivocó por falta de inteligencia: se equivocó porque le dimos un encargo que
no podía cuestionar.